# Fall Detection Inference Demo

This notebook demonstrates how to use a trained fall detection model on a **single 60-second IMU packet** -- simulating what the cloud service does for each incoming packet from a wearable.

## Prerequisites

1. Run `fall_detection_train.ipynb` first to produce `fall_detection_model.pt`
2. Either:
   - Upload `fall_detection_model.pt` to Colab (Files tab on the left), OR
   - Mount Google Drive and load from there

## What this notebook does

1. Loads a trained checkpoint (model weights + normalization stats + operating threshold)
2. Defines a `predict(window)` function that takes a `(3000, 6)` IMU window and returns fall probability
3. Runs inference on synthetic example windows to demonstrate the API

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 1. Model architecture

This must match the architecture used during training. It's the same `CNN1D` class from the training notebook.

In [ ]:
class CNN1D(nn.Module):
    def __init__(self, in_channels=6, channels=None, kernel_sizes=None, dropout=0.3):
        super().__init__()
        channels = channels or [32, 64, 128, 128]
        kernel_sizes = kernel_sizes or [7, 5, 5, 3]

        layers = []
        prev_ch = in_channels
        for ch, ks in zip(channels, kernel_sizes):
            layers.extend([
                nn.Conv1d(prev_ch, ch, kernel_size=ks, stride=2, padding=ks // 2),
                nn.BatchNorm1d(ch),
                nn.ReLU(inplace=True),
            ])
            prev_ch = ch

        self.features = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(channels[-1], 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).squeeze(-1)
        return self.classifier(x)

## 2. Load the checkpoint

Point `CHECKPOINT_PATH` to wherever you uploaded `fall_detection_model.pt`.

In [ ]:
# Option A: File uploaded directly to Colab runtime
CHECKPOINT_PATH = "fall_detection_model.pt"

# Option B: Mount Google Drive and load from there
# from google.colab import drive
# drive.mount("/content/drive")
# CHECKPOINT_PATH = "/content/drive/MyDrive/fall_detection_model.pt"

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)

config = checkpoint["config"]
norm_stats = checkpoint["norm_stats"]
best_threshold = checkpoint["best_threshold"]
test_metrics = checkpoint["test_metrics"]

print("Checkpoint loaded.")
print(f"  Best threshold: {best_threshold:.3f}")
print(f"  Test TPR:       {test_metrics['tpr']:.3f}")
print(f"  Test FPR:       {test_metrics['fpr']:.3f}")
print(f"  Test AUC:       {test_metrics['roc_auc']:.4f}")
print(f"  Norm mean:      {[f'{m:+.3f}' for m in norm_stats['mean']]}")
print(f"  Norm std:       {[f'{s:+.3f}' for s in norm_stats['std']]}")

# Rebuild model and load weights
model = CNN1D(
    in_channels=6,
    channels=config["model"]["channels"],
    kernel_sizes=config["model"]["kernel_sizes"],
    dropout=config["model"]["dropout"],
).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("\nModel is ready for inference.")

## 3. The `predict` function

This is what the cloud service would call for each incoming 60s packet from the gateway.

Input: numpy array of shape `(3000, 6)` with columns `[ax, ay, az, gx, gy, gz]`
- `ax, ay, az`: accelerometer in g
- `gx, gy, gz`: gyroscope angular velocity in deg/s
- 3000 samples = 60 seconds at 50 Hz

Output: dict with `is_fall`, `confidence`, `logit`, `latency_ms`

In [ ]:
import time


def predict(window: np.ndarray, model, norm_stats, threshold):
    """Run fall detection on a single 60-second window.

    Args:
        window: np.ndarray of shape (3000, 6) -- raw IMU data at 50 Hz
                Columns: [ax, ay, az, gx, gy, gz]
                Units:   accel in g, gyro in deg/s
        model: trained CNN1D in eval mode
        norm_stats: dict with 'mean' and 'std' (6-element lists from training)
        threshold: classification threshold (picked during training on val set)

    Returns:
        dict with keys:
            is_fall (bool):    prediction
            confidence (float): sigmoid probability in [0, 1]
            logit (float):      raw logit (pre-sigmoid)
            latency_ms (float): inference time in milliseconds
    """
    # 1) Normalize to match training distribution
    mean = np.array(norm_stats["mean"], dtype=np.float32)
    std = np.array(norm_stats["std"], dtype=np.float32)
    normed = (window.astype(np.float32) - mean) / std

    # 2) Convert to channels-first tensor with batch dim: (1, 6, 3000)
    x = torch.from_numpy(normed.T).float().unsqueeze(0).to(DEVICE)

    # 3) Forward pass
    t0 = time.perf_counter()
    with torch.no_grad():
        logit = model(x).item()
    latency_ms = (time.perf_counter() - t0) * 1000

    confidence = float(torch.sigmoid(torch.tensor(logit)))

    return {
        "is_fall": confidence >= threshold,
        "confidence": confidence,
        "logit": logit,
        "latency_ms": latency_ms,
    }


print("predict() function defined")

## 4. Demo on real test windows

The training notebook bundles one fall and one ADL window from the held-out test set into the checkpoint (stored in raw units -- NOT pre-normalized). We use those here to demonstrate the full preprocessing + inference pipeline on realistic data.

If the bundled windows are missing (older checkpoint), we fall back to a synthetic demo.

In [ ]:
WINDOW_SAMPLES = 3000  # 60s @ 50Hz
N_CHANNELS = 6

sample_windows = checkpoint.get("sample_windows", {})

if "fall" in sample_windows and "adl" in sample_windows:
    # Use real bundled test windows
    adl_info = sample_windows["adl"]
    fall_info = sample_windows["fall"]

    adl_window = np.array(adl_info["data"], dtype=np.float32)
    fall_window = np.array(fall_info["data"], dtype=np.float32)

    print("Using real test windows from checkpoint:")
    print(f"  ADL:  dataset={adl_info['dataset']}, subject={adl_info['subject_id']}, activity={adl_info['activity']} (valid {adl_info['valid_length']}/{WINDOW_SAMPLES} samples)")
    print(f"  Fall: dataset={fall_info['dataset']}, subject={fall_info['subject_id']}, activity={fall_info['activity']}, type={fall_info['fall_type']} (valid {fall_info['valid_length']}/{WINDOW_SAMPLES} samples)")
    using_real_data = True
else:
    # Fallback: synthetic data
    print("No bundled sample windows found. Generating synthetic demo data.")
    print("(Re-run the training notebook to bundle real windows in the checkpoint.)")
    np.random.seed(0)
    adl_window = np.random.randn(WINDOW_SAMPLES, N_CHANNELS).astype(np.float32) * 0.1
    adl_window[:, 2] += 1.0  # gravity

    fall_window = np.random.randn(WINDOW_SAMPLES, N_CHANNELS).astype(np.float32) * 0.1
    fall_window[:, 2] += 1.0
    fall_window[1400:1430, :3] += np.random.randn(30, 3).astype(np.float32) * 3.0
    fall_window[1400:1430, 3:] += np.random.randn(30, 3).astype(np.float32) * 100.0
    using_real_data = False

# Run inference on both windows
result_adl = predict(adl_window, model, norm_stats, best_threshold)
result_fall = predict(fall_window, model, norm_stats, best_threshold)

print()
print(f"ADL window:  confidence={result_adl['confidence']:.3f}  is_fall={result_adl['is_fall']}  ({result_adl['latency_ms']:.1f} ms)")
print(f"Fall window: confidence={result_fall['confidence']:.3f}  is_fall={result_fall['is_fall']}  ({result_fall['latency_ms']:.1f} ms)")
print(f"\nOperating threshold: {best_threshold:.3f}")

## 5. Visualize the two windows

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 6), sharex=True)
time_axis = np.arange(WINDOW_SAMPLES) / 50.0  # seconds

source_tag = "real test data" if using_real_data else "synthetic"

# Accel row
axes[0, 0].plot(time_axis, adl_window[:, :3])
axes[0, 0].set_title(f"ADL window ({source_tag}) | confidence={result_adl['confidence']:.3f}")
axes[0, 0].set_ylabel("Accel (g)")
axes[0, 0].legend(["ax", "ay", "az"], loc="upper right", fontsize=8)
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(time_axis, fall_window[:, :3])
axes[0, 1].set_title(f"Fall window ({source_tag}) | confidence={result_fall['confidence']:.3f}")
axes[0, 1].legend(["ax", "ay", "az"], loc="upper right", fontsize=8)
axes[0, 1].grid(alpha=0.3)

# Gyro row
axes[1, 0].plot(time_axis, adl_window[:, 3:])
axes[1, 0].set_xlabel("Time (s)")
axes[1, 0].set_ylabel("Gyro (deg/s)")
axes[1, 0].legend(["gx", "gy", "gz"], loc="upper right", fontsize=8)
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(time_axis, fall_window[:, 3:])
axes[1, 1].set_xlabel("Time (s)")
axes[1, 1].legend(["gx", "gy", "gz"], loc="upper right", fontsize=8)
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Latency benchmark

Measure end-to-end inference latency across many windows. This is what the cloud service would see per packet.

Target: <= 1 second per packet (comfortable for a 60s-cadence system).

In [ ]:
# Warm up
for _ in range(3):
    _ = predict(adl_window, model, norm_stats, best_threshold)

# Benchmark
n_runs = 50
latencies = []
for _ in range(n_runs):
    r = predict(adl_window, model, norm_stats, best_threshold)
    latencies.append(r["latency_ms"])

latencies = np.array(latencies)
print(f"Inference latency over {n_runs} runs:")
print(f"  mean:   {latencies.mean():.2f} ms")
print(f"  median: {np.median(latencies):.2f} ms")
print(f"  p95:    {np.percentile(latencies, 95):.2f} ms")
print(f"  min:    {latencies.min():.2f} ms")
print(f"  max:    {latencies.max():.2f} ms")

## Integration notes

To plug this into the cloud service:

1. Load the checkpoint **once** at service startup
2. Call `predict(window, model, norm_stats, best_threshold)` per incoming packet
3. The gateway must send windows in the same format: `(3000, 6)` numpy array, 50 Hz, accel in g, gyro in deg/s
4. If the wearable samples at a different rate, resample to 50 Hz before calling `predict`
5. If a fall is detected, route the alert to nursing staff with the confidence score and timestamp

The model is deterministic given fixed weights and no dropout (`model.eval()`), so the same window always produces the same prediction.